# Notebook to forecast Support tickets

#### Imports & others

In [1]:
import os
import getpass
home_dir = os.path.expanduser("~")
username = getpass.getuser()
print(home_dir)
print(username)

/home/bigdata
catarinap


In [2]:
!uv pip install --target ./mypkg --only-binary :all: "pandas<2.0" prophet==1.2.1

Using CPython 3.11.11 interpreter at: venv/bin/python3
Resolved 19 packages in 3.43s                                        
⠙ Preparing packages... (0/19)                                                  
⠙ Preparing packages... (0/19)------------------     0 B/119.90 KiB          
⠙ Preparing packages... (0/19)------------------ 3.08 KiB/119.90 KiB         
⠙ Preparing packages... (0/19)------------------ 3.08 KiB/119.90 KiB         
pyparsing            ------------------------------ 3.08 KiB/119.90 KiB
⠙ Preparing packages... (0/19)------------------     0 B/11.56 MiB           
pyparsing            ------------------------------ 3.08 KiB/119.90 KiB
⠙ Preparing packages... (0/19)------------------ 2.97 KiB/11.56 MiB          
pyparsing            ------------------------------ 3.08 KiB/119.90 KiB
⠙ Preparing packages... (0/19)------------------ 2.97 KiB/11.56 MiB          
pyparsing            ------------------------------ 3.08 KiB/119.90 KiB
⠙ Preparing packages... (0/19)--------

In [3]:
import sys
import os

# 1. Point Python to your custom folder
path = os.path.abspath('./mypkg')
if path not in sys.path:
    sys.path.insert(0, path)

# 2. Verify the folder actually contains prophet
if not os.path.exists(os.path.join(path, "prophet")):
    print(f"ERROR: 'prophet' is not in {path}. The installation likely failed.")
else:
    from prophet import Prophet
    print("Success: Prophet imported from ./mypkg")

[D 2026-05-07 10:56:31,672.672 matplotlib] matplotlib data path: /home/bigdata/mypkg/matplotlib/mpl-data
[D 2026-05-07 10:56:31,680.680 matplotlib] CONFIGDIR=/home/bigdata/.config/matplotlib
[D 2026-05-07 10:56:31,699.699 matplotlib] interactive is False
[D 2026-05-07 10:56:31,699.699 matplotlib] platform is linux
[D 2026-05-07 10:56:31,900.900 matplotlib] CACHEDIR=/home/bigdata/.cache/matplotlib
[D 2026-05-07 10:56:31,900.900 matplotlib.font_manager] font search path [PosixPath('/home/bigdata/mypkg/matplotlib/mpl-data/fonts/ttf'), PosixPath('/home/bigdata/mypkg/matplotlib/mpl-data/fonts/afm'), PosixPath('/home/bigdata/mypkg/matplotlib/mpl-data/fonts/pdfcorefonts')]
[I 2026-05-07 10:56:32,686.686 matplotlib.font_manager] generated new fontManager


Success: Prophet imported from ./mypkg


In [28]:
from prophet import Prophet
import pandas as pd
import numpy as np

In [29]:
import importlib.metadata
print(importlib.metadata.version("prophet"))

1.2.1


In [30]:
!hdfs dfs -ls /user/{username}/notebooks/Workload\ Projections

Found 28 items
drwx------   - catarinap hdfs          0 2026-03-23 11:09 /user/catarinap/notebooks/Workload Projections/.ipynb_checkpoints
-rw-r--r--   3 catarinap hdfs  108114332 2026-03-23 09:49 /user/catarinap/notebooks/Workload Projections/181797_part_aa.csv
-rw-r--r--   3 catarinap hdfs  107778046 2026-03-23 09:48 /user/catarinap/notebooks/Workload Projections/181797_part_ab.csv
-rw-------   3 catarinap hdfs    8334793 2025-09-08 13:16 /user/catarinap/notebooks/Workload Projections/Adyen 2025 Calls.csv
-rw-------   3 catarinap hdfs   13301199 2025-09-12 11:22 /user/catarinap/notebooks/Workload Projections/Aircall_Combined_Detailed.csv
-rw-r--r--   3 catarinap hdfs     177186 2026-03-23 11:07 /user/catarinap/notebooks/Workload Projections/Data Processing.ipynb
drwxr-xr-x   - catarinap hdfs          0 2026-03-12 14:14 /user/catarinap/notebooks/Workload Projections/Forecasted data
-rw-r--r--   3 catarinap hdfs      61456 2026-03-23 11:09 /user/catarinap/notebooks/Workload Projections

## Read data

In [31]:
!hdfs dfs -copyToLocal /user/{username}/notebooks/Workload\ Projections/aggregated_cases_df.parquet

copyToLocal: `aggregated_cases_df.parquet/_SUCCESS': File exists
copyToLocal: `aggregated_cases_df.parquet/part-00000-19791f14-46a8-45a8-ba04-5cb4c2d37cf3-c000.zstd.parquet': File exists
copyToLocal: `aggregated_cases_df.parquet/part-00001-19791f14-46a8-45a8-ba04-5cb4c2d37cf3-c000.zstd.parquet': File exists
copyToLocal: `aggregated_cases_df.parquet/part-00002-19791f14-46a8-45a8-ba04-5cb4c2d37cf3-c000.zstd.parquet': File exists
copyToLocal: `aggregated_cases_df.parquet/part-00003-19791f14-46a8-45a8-ba04-5cb4c2d37cf3-c000.zstd.parquet': File exists
copyToLocal: `aggregated_cases_df.parquet/part-00004-19791f14-46a8-45a8-ba04-5cb4c2d37cf3-c000.zstd.parquet': File exists
copyToLocal: `aggregated_cases_df.parquet/part-00005-19791f14-46a8-45a8-ba04-5cb4c2d37cf3-c000.zstd.parquet': File exists
copyToLocal: `aggregated_cases_df.parquet/part-00006-19791f14-46a8-45a8-ba04-5cb4c2d37cf3-c000.zstd.parquet': File exists
copyToLocal: `aggregated_cases_df.parquet/part-00007-19791f14-46a8-45a8-ba04-5cb4

In [32]:
tickets = pd.read_parquet('aggregated_cases_df.parquet').drop_duplicates()
tickets.head()

,month,domain,subdomain,team,support_team,origin,category,unique_id_count
0,2024-02-01,Support,Technical Support,Technical Support,Ecom Cards,Email,Service,864
1,2024-09-01,Support,Operational Support,Operational Support,Operational Support,Email,Internal,990
2,2025-01-01,Support,Technical Support,Account Setup Operations,Account Setup Operations,Webform,Service,394
3,2025-10-01,Support,Technical Support,Technical Support,LATAM - Technical Support,Email,Service,588
4,2025-11-01,Support,Operational Support,Operational Support,Operational Support,Email,Service,4349


In [33]:
tickets.to_csv('tickets.csv', index=False)

In [34]:
!hdfs dfs -put -f tickets.csv /user/{username}/notebooks/adyen_downloader

In [35]:
tickets['support_team'].unique()

array(['Ecom Cards', 'Operational Support', 'Account Setup Operations',
       'LATAM - Technical Support', 'Disputes', None, 'Ecom APM',
       'Finance', 'Japan', 'IPP Core', 'IPP Adv. Payments',
       'India - Technical Support', 'IPP Integrations', 'Partner Support',
       'Ecom Core', 'Platforms', 'Ecom Integration',
       'LATAM - Operational Support', 'China'], dtype=object)

In [36]:
!hdfs dfs -copyToLocal /user/{username}/notebooks/Workload\ Projections/active_account_summary_df.parquet

In [37]:
accounts = pd.read_parquet('active_account_summary_df.parquet')
accounts.head()

,month_seq,region,managed_region,pillar_clean,unique_accounts
0,2013-05-01,EMEA,EMEA,Digital,71
1,2013-11-01,EMEA,None,Digital,383
2,2022-08-01,EMEA,None,Platforms,91
3,2025-06-01,APAC,None,Digital,435
4,2024-08-01,APAC,None,None,4


In [38]:
accounts.to_csv('accounts.csv', index=False)

In [39]:
!hdfs dfs -put -f accounts.csv /user/{username}/notebooks/adyen_downloader

In [40]:
grouped = accounts.groupby('month_seq')['unique_accounts'].sum().reset_index()
print(grouped)

      month_seq  unique_accounts
0    2010-06-01                1
1    2010-07-01                8
2    2010-08-01               14
3    2010-09-01               30
4    2010-10-01               46
..          ...              ...
185  2025-11-01             9676
186  2025-12-01             9728
187  2026-01-01             9750
188  2026-02-01             9788
189  2026-03-01             9798

[190 rows x 2 columns]


In [41]:
!hdfs dfs -copyToLocal /user/{username}/notebooks/Workload\ Projections/clean_calls_df.parquet

copyToLocal: `clean_calls_df.parquet/_SUCCESS': File exists
copyToLocal: `clean_calls_df.parquet/part-00000-1050be65-241f-441a-9e14-458358e98323-c000.zstd.parquet': File exists
copyToLocal: `clean_calls_df.parquet/part-00001-1050be65-241f-441a-9e14-458358e98323-c000.zstd.parquet': File exists
copyToLocal: `clean_calls_df.parquet/part-00002-1050be65-241f-441a-9e14-458358e98323-c000.zstd.parquet': File exists
copyToLocal: `clean_calls_df.parquet/part-00003-1050be65-241f-441a-9e14-458358e98323-c000.zstd.parquet': File exists
copyToLocal: `clean_calls_df.parquet/part-00004-1050be65-241f-441a-9e14-458358e98323-c000.zstd.parquet': File exists
copyToLocal: `clean_calls_df.parquet/part-00005-1050be65-241f-441a-9e14-458358e98323-c000.zstd.parquet': File exists
copyToLocal: `clean_calls_df.parquet/part-00006-1050be65-241f-441a-9e14-458358e98323-c000.zstd.parquet': File exists
copyToLocal: `clean_calls_df.parquet/part-00007-1050be65-241f-441a-9e14-458358e98323-c000.zstd.parquet': File exists
copy

In [42]:
calls = pd.read_parquet('clean_calls_df.parquet')
calls.head()

,month,support_team,calls
0,2023-01-01,Japan,157
1,2023-01-01,LATAM,100
2,2023-01-01,Operational Support,1137
3,2023-02-01,Japan,167
4,2023-02-01,LATAM,107


In [43]:
calls['month'].unique()

array(['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01',
       '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01',
       '2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01',
       '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01',
       '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01',
       '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01',
       '2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01',
       '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01',
       '2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01',
       '2026-01-01', '2026-02-01', '2026-03-01'], dtype=object)

In [44]:
calls.to_csv('clean_calls_df.csv', index=False)

In [45]:
!hdfs dfs -put -f clean_calls_df.csv /user/{username}/notebooks/adyen_downloader

## Account Growth

In [46]:
!uv pip install --target ./mypkg statsmodels

Resolved 9 packages in 293ms                                         
⠙ Preparing packages... (0/3)                                                   
⠙ Preparing packages... (0/3)----     0 B/233.30 kB                     
⠙ Preparing packages... (0/3)---- 3.64 kB/233.30 kB                     
⠙ Preparing packages... (0/3)---- 7.74 kB/233.30 kB                     
⠙ Preparing packages... (0/3)---- 11.83 kB/233.30 kB                    
⠙ Preparing packages... (0/3)---- 15.93 kB/233.30 kB                    
⠙ Preparing packages... (0/3)---- 20.02 kB/233.30 kB                    
⠙ Preparing packages... (0/3)---- 28.21 kB/233.30 kB                    
⠙ Preparing packages... (0/3)---- 40.50 kB/233.30 kB                    
⠙ Preparing packages... (0/3)---- 44.60 kB/233.30 kB                    
⠙ Preparing packages... (0/3)---- 56.89 kB/233.30 kB                    
⠙ Preparing packages... (0/3)---- 69.17 kB/233.30 kB                    
⠙ Preparing packages... (0/3)---- 85.50 kB/233

In [47]:
import matplotlib.pyplot as plt
from datetime import datetime
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

In [48]:
current_month_start = datetime.today().replace(day=1).date()
current_month_start

datetime.date(2026, 3, 1)

In [49]:
accounts['month_seq'] = pd.to_datetime(accounts['month_seq']).dt.date

In [50]:
latam_accounts = accounts[(accounts['region'] == 'LATAM') & (accounts['month_seq'] < current_month_start)].copy()
latam_accounts['month'] = latam_accounts['month_seq']
latam_accounts.head()

,month_seq,region,managed_region,pillar_clean,unique_accounts,month
8,2018-05-01,LATAM,None,None,25,2018-05-01
9,2018-04-01,LATAM,LATAM,Digital,35,2018-04-01
15,2021-05-01,LATAM,LATAM,UC,19,2021-05-01
27,2015-11-01,LATAM,None,UC,2,2015-11-01
28,2017-02-01,LATAM,LATAM,None,6,2017-02-01


In [51]:
non_latam_accounts = accounts[(accounts['region'] != 'LATAM') & (accounts['month_seq'] < current_month_start)].copy()
non_latam_accounts['month'] = non_latam_accounts['month_seq']
non_latam_accounts.head()

,month_seq,region,managed_region,pillar_clean,unique_accounts,month
0,2013-05-01,EMEA,EMEA,Digital,71,2013-05-01
1,2013-11-01,EMEA,None,Digital,383,2013-11-01
2,2022-08-01,EMEA,None,Platforms,91,2022-08-01
3,2025-06-01,APAC,None,Digital,435,2025-06-01
4,2024-08-01,APAC,None,None,4,2024-08-01


In [52]:
non_latam_accounts = non_latam_accounts.groupby('month').agg({'unique_accounts': 'sum'})
print(non_latam_accounts.sort_values(by='month', ascending=False))

            unique_accounts
month                      
2026-02-01             9468
2026-01-01             9431
2025-12-01             9406
2025-11-01             9359
2025-10-01             9294
...                     ...
2010-10-01               46
2010-09-01               30
2010-08-01               14
2010-07-01                8
2010-06-01                1

[189 rows x 1 columns]


Test if data is stationary

In [53]:
ts = pd.Series(
    non_latam_accounts['unique_accounts'].values,
    index=non_latam_accounts.index
)
log_ts = np.log(ts + 1)
diff_log_ts = log_ts.diff().dropna()

In [54]:
adf_result = adfuller(diff_log_ts)
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])

ADF Statistic: -2.906222277809479
p-value: 0.044629201036050906


Model fitting

In [55]:
model = ARIMA(log_ts, order=(1, 1, 1)) 
fit_model = model.fit()

/home/bigdata/venv/lib64/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/bigdata/venv/lib64/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/bigdata/venv/lib64/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [56]:
print(fit_model.summary())

                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                  189
Model:                 ARIMA(1, 1, 1)   Log Likelihood                 198.080
Date:                Mon, 23 Mar 2026   AIC                           -390.161
Time:                        11:09:57   BIC                           -380.451
Sample:                    06-01-2010   HQIC                          -386.227
                         - 02-01-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.9911      0.003    393.103      0.000       0.986       0.996
ma.L1         -0.2094      0.054     -3.910      0.000      -0.314      -0.104
sigma2         0.0070      0.000     38.026      0.0

In [57]:
forecast_steps = 12
forecast = fit_model.forecast(steps=forecast_steps)
print(forecast)

2026-03-01    9.159505
2026-04-01    9.163198
2026-05-01    9.166858
2026-06-01    9.170486
2026-07-01    9.174082
2026-08-01    9.177646
2026-09-01    9.181179
2026-10-01    9.184680
2026-11-01    9.188150
2026-12-01    9.191589
2027-01-01    9.194998
2027-02-01    9.198376
Freq: MS, Name: predicted_mean, dtype: float64


In [58]:
forecast_exp = np.exp(forecast) - 1  
print(forecast_exp)

2026-03-01    9503.348897
2026-04-01    9538.514762
2026-05-01    9573.497393
2026-06-01    9608.296611
2026-07-01    9642.912259
2026-08-01    9677.344204
2026-09-01    9711.592335
2026-10-01    9745.656561
2026-11-01    9779.536814
2026-12-01    9813.233047
2027-01-01    9846.745234
2027-02-01    9880.073368
Freq: MS, Name: predicted_mean, dtype: float64


In [59]:
future_months = pd.date_range(start=current_month_start, periods=12, freq='MS')
print(future_months)

DatetimeIndex(['2026-03-01', '2026-04-01', '2026-05-01', '2026-06-01',
               '2026-07-01', '2026-08-01', '2026-09-01', '2026-10-01',
               '2026-11-01', '2026-12-01', '2027-01-01', '2027-02-01'],
              dtype='datetime64[ns]', freq='MS')


In [60]:
forecast_df = pd.DataFrame({
    'month': future_months,
    'unique_accounts': forecast_exp.values,
    'type': 'Forecast'
})
print(forecast_df)

        month  unique_accounts      type
0  2026-03-01      9503.348897  Forecast
1  2026-04-01      9538.514762  Forecast
2  2026-05-01      9573.497393  Forecast
3  2026-06-01      9608.296611  Forecast
4  2026-07-01      9642.912259  Forecast
5  2026-08-01      9677.344204  Forecast
6  2026-09-01      9711.592335  Forecast
7  2026-10-01      9745.656561  Forecast
8  2026-11-01      9779.536814  Forecast
9  2026-12-01      9813.233047  Forecast
10 2027-01-01      9846.745234  Forecast
11 2027-02-01      9880.073368  Forecast


In [61]:
actuals_df = non_latam_accounts.copy()
actuals_df = actuals_df.reset_index()  
actuals_df['type'] = 'Actual'
print(actuals_df)

          month  unique_accounts    type
0    2010-06-01                1  Actual
1    2010-07-01                8  Actual
2    2010-08-01               14  Actual
3    2010-09-01               30  Actual
4    2010-10-01               46  Actual
..          ...              ...     ...
184  2025-10-01             9294  Actual
185  2025-11-01             9359  Actual
186  2025-12-01             9406  Actual
187  2026-01-01             9431  Actual
188  2026-02-01             9468  Actual

[189 rows x 3 columns]


In [62]:
actuals_df['month'] = pd.to_datetime(actuals_df['month'])
forecast_df['month'] = pd.to_datetime(forecast_df['month'])
combined_df = pd.concat([actuals_df, forecast_df])
print(combined_df)

        month  unique_accounts      type
0  2010-06-01         1.000000    Actual
1  2010-07-01         8.000000    Actual
2  2010-08-01        14.000000    Actual
3  2010-09-01        30.000000    Actual
4  2010-10-01        46.000000    Actual
..        ...              ...       ...
7  2026-10-01      9745.656561  Forecast
8  2026-11-01      9779.536814  Forecast
9  2026-12-01      9813.233047  Forecast
10 2027-01-01      9846.745234  Forecast
11 2027-02-01      9880.073368  Forecast

[201 rows x 3 columns]


In [63]:
combined_df.to_csv('non_latam_accounts_forecast.csv', index=False)

In [64]:
!hdfs dfs -put -f non_latam_accounts_forecast.csv /user/{username}/notebooks/Workload\ Projections/Forecasted\ data

In [65]:
!hdfs dfs -put -f non_latam_accounts_forecast.csv /user/{username}/notebooks/adyen_downloader

## Squads Forecasting

### Service cases

In [66]:
def create_service_cases_df(df):

    df['month'] = pd.to_datetime(df['month'])

    filtered_df = df[
        (df['month'].dt.date < current_month_start) &
        (df['month'].dt.date >= pd.to_datetime('2023-09-01').date()) &
        (~df['origin'].isin(['Side Conversation', 'Call'])) &
        (df['category'] == 'Service')
    ]

    service_df = (
        filtered_df
        .groupby(['month', 'support_team'], as_index=False)
        .agg(cases=('unique_id_count', 'sum'))
        .dropna()
    )

    service_df = service_df.rename(columns={
        'month': 'month',
        'support_team': 'squad',
        'cases': 'cases'
    })

    return service_df

In [67]:
service_df = create_service_cases_df(tickets)
print(service_df)

         month                        squad  cases
0   2023-09-01     Account Setup Operations   3031
1   2023-09-01                     Disputes   1561
2   2023-09-01                     Ecom APM    736
3   2023-09-01                   Ecom Cards   1045
4   2023-09-01                    Ecom Core   2300
..         ...                          ...    ...
503 2026-02-01  LATAM - Operational Support   1358
504 2026-02-01    LATAM - Technical Support    442
505 2026-02-01          Operational Support   5942
506 2026-02-01              Partner Support    208
507 2026-02-01                    Platforms    976

[508 rows x 3 columns]


In [68]:
def prepare_forecast_df(service_df, account_growth):
    df = (
        service_df
        .merge(account_growth, on='month', how='left')
        .rename(columns={'month': 'ds', 'cases': 'y'})
        .dropna(subset=['ds', 'y', 'unique_accounts'])
    )
    return df

In [69]:
test = prepare_forecast_df(service_df, combined_df)
print(test)

            ds                        squad     y  unique_accounts    type
0   2023-09-01     Account Setup Operations  3031           7899.0  Actual
1   2023-09-01                     Disputes  1561           7899.0  Actual
2   2023-09-01                     Ecom APM   736           7899.0  Actual
3   2023-09-01                   Ecom Cards  1045           7899.0  Actual
4   2023-09-01                    Ecom Core  2300           7899.0  Actual
..         ...                          ...   ...              ...     ...
503 2026-02-01  LATAM - Operational Support  1358           9468.0  Actual
504 2026-02-01    LATAM - Technical Support   442           9468.0  Actual
505 2026-02-01          Operational Support  5942           9468.0  Actual
506 2026-02-01              Partner Support   208           9468.0  Actual
507 2026-02-01                    Platforms   976           9468.0  Actual

[508 rows x 5 columns]


In [70]:
nested_df = {
    squad: df.drop(columns='squad')
    for squad, df in test.groupby('squad')
}
print(nested_df)

{'Account Setup Operations':             ds     y  unique_accounts    type
0   2023-09-01  3031           7899.0  Actual
14  2023-10-01  3042           7963.0  Actual
29  2023-11-01  3140           8044.0  Actual
44  2023-12-01  2198           8095.0  Actual
59  2024-01-01  2720           8123.0  Actual
73  2024-02-01  2535           8157.0  Actual
88  2024-03-01  2636           8209.0  Actual
104 2024-04-01  2868           8300.0  Actual
120 2024-05-01  2858           8389.0  Actual
136 2024-06-01  2622           8436.0  Actual
152 2024-07-01  2843           8488.0  Actual
168 2024-08-01  2530           8503.0  Actual
185 2024-09-01  2832           8569.0  Actual
202 2024-10-01  3363           8647.0  Actual
220 2024-11-01  3066           8711.0  Actual
238 2024-12-01  2397           8756.0  Actual
256 2025-01-01  2754           8821.0  Actual
274 2025-02-01  2662           8865.0  Actual
292 2025-03-01  2955           8945.0  Actual
310 2025-04-01  2924           8998.0  Actual
328 2

In [71]:
def run_service_prophet(df, account_growth):
    
     # Ensure required columns are present
    required_cols = {'ds', 'y', 'unique_accounts'}
    if not required_cols.issubset(df.columns):
        return None
    
    # Remove NAs and check row count
    df = df.dropna(subset=['ds', 'y', 'unique_accounts'])
    if len(df) < 10:
        return None  # Prophet requires enough data points
    
    # Fit the model with regressor
    m = Prophet()
    m.add_regressor('unique_accounts', prior_scale=0.1)
    m.fit(df)
    
    # Make future df
    future = m.make_future_dataframe(periods=13, freq='M')
    future = future.drop_duplicates(subset=['ds'])
    future['ds'] = future['ds'].values.astype('datetime64[M]')
    
    # Ensure account_growth has no NaNs
    account_growth = account_growth.dropna(subset=['month', 'unique_accounts'])
    account_growth = account_growth.rename(columns={'month': 'ds'})
    
    # Ensure datetime format matches for merging
    future['ds'] = pd.to_datetime(future['ds'])
    account_growth['ds'] = pd.to_datetime(account_growth['ds'])
    
    # Join with account_growth on 'ds'
    future = future.merge(account_growth, on='ds', how='left')
    future['unique_accounts'] = future['unique_accounts'].fillna(method='ffill')
    future['unique_accounts'] = future['unique_accounts'].fillna(0)
    
    future = future[['ds', 'unique_accounts']]
    
    # Forecast
    forecast = m.predict(future)
    last_train_date = df['ds'].max()
    forecast = forecast[forecast['ds'] > last_train_date]
    
    # Model evaluation on historical part
    comparison = df.merge(forecast[['ds', 'yhat']], on='ds', how='left').dropna()
    comparison['residuals'] = comparison['y'] - comparison['yhat']

    ss_res = np.sum(comparison['residuals'] ** 2)
    ss_tot = np.sum((comparison['y'] - np.mean(comparison['y'])) ** 2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else None

    mape = np.mean(np.abs((comparison['y'] - comparison['yhat']) / comparison['y'])) * 100

    return {
        'model': m,
        'forecast': forecast,
        'r_squared': r_squared,
        'mape': mape
    }


In [72]:
results = {}
for squad, df in nested_df.items():
    result = run_service_prophet(df, combined_df)
    if result is not None:
        results[squad] = result
print(results)

[I 2026-03-23 11:10:07,669.669 prophet] Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
[I 2026-03-23 11:10:07,669.669 prophet] Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
[I 2026-03-23 11:10:07,698.698 prophet] n_changepoints greater than number of observations. Using 23.
[D 2026-03-23 11:10:07,702.702 cmdstanpy] input tempfile: /tmp/tmpy_ntexaz/8vu_pth8.json
[D 2026-03-23 11:10:07,705.705 cmdstanpy] input tempfile: /tmp/tmpy_ntexaz/njmr_18t.json
[D 2026-03-23 11:10:07,707.707 cmdstanpy] idx 0
[D 2026-03-23 11:10:07,707.707 cmdstanpy] running CmdStan, num_threads: None
[D 2026-03-23 11:10:07,707.707 cmdstanpy] CmdStan args: ['/home/bigdata/mypkg/prophet/stan_model/prophet_model.bin', 'random', 'seed=2222', 'data', 'file=/tmp/tmpy_ntexaz/8vu_pth8.json', 'init=/tmp/tmpy_ntexaz/njmr_18t.json', 'output', 'file=/tmp/tmpy_ntexaz/prophet_modeleqtktwr7/prophet_model-20260323111007.csv', 'method=optimize', 'alg

{'Account Setup Operations': {'model': <prophet.forecaster.Prophet object at 0x7efa386391d0>, 'forecast':            ds        trend   yhat_lower   yhat_upper  trend_lower  \
31 2026-03-01  1873.359082  2675.304955  2834.773309  1872.375479   
32 2026-04-01  1800.256866  2333.970864  2491.737127  1797.240652   
33 2026-05-01  1729.512786  2512.650041  2667.923789  1723.803963   
34 2026-06-01  1656.410570  2489.427361  2645.595738  1646.515472   
35 2026-07-01  1585.666489  2660.392726  2821.882036  1571.720382   
36 2026-08-01  1512.564273  1876.589547  2044.813214  1494.223426   
37 2026-09-01  1439.462057  2496.925161  2670.236742  1416.607758   
38 2026-10-01  1368.717976  2629.485984  2800.286277  1341.240342   
39 2026-11-01  1295.615760  2474.714682  2645.892338  1262.190313   
40 2026-12-01  1224.871680  1703.414878  1882.773298  1184.812484   
41 2027-01-01  1151.769464  2027.027992  2218.533464  1104.713358   
42 2027-02-01  1078.667247  1900.847264  2100.782590  1024.961190 

In [73]:
# Get current timestamp
current_timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Prepare lists to collect results for each component
forecast_list = []
r_squared_list = []
mape_list = []

# Loop through the results and gather data
for squad, result in results.items():
    # Add forecast data (ds and yhat)
    forecast_df = result['forecast'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    forecast_df[['yhat', 'yhat_lower', 'yhat_upper']] = forecast_df[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
    forecast_df['squad'] = squad  # Add the squad name for identification
    forecast_df['timestamp'] = current_timestamp  # Add timestamp
    forecast_df['category'] = 'Service'  # Add category column
    forecast_list.append(forecast_df)
    
    # Add R-squared data
    r_squared_list.append({'squad': squad, 'r_squared': result['r_squared'], 'timestamp': current_timestamp})
    
    # Add MAPE data
    mape_list.append({'squad': squad, 'mape': result['mape'], 'timestamp': current_timestamp})

# Combine the lists into DataFrames
forecast_combined = pd.concat(forecast_list, ignore_index=True)
r_squared_combined = pd.DataFrame(r_squared_list)
mape_combined = pd.DataFrame(mape_list)

# Ensure correct column names
forecast_combined.columns = ['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'squad', 'timestamp', 'category']

# Save the combined data to CSV files
forecast_combined.to_csv('service_combined_forecast.csv', mode='w', header=True, index=False)
r_squared_combined.to_csv('service_combined_r_squared.csv', mode='w', header=True, index=False)
mape_combined.to_csv('service_combined_mape.csv', mode='w', header=True, index=False)

print(f"All results have been saved to service_combined_forecast.csv, service_combined_r_squared.csv, and service_combined_mape.csv with timestamp {current_timestamp}.")

All results have been saved to service_combined_forecast.csv, service_combined_r_squared.csv, and service_combined_mape.csv with timestamp 2026-03-23 11:10:13.


In [74]:
!hdfs dfs -put -f service_combined_forecast.csv /user/{username}/notebooks/Workload\ Projections/Forecasted\ data

In [75]:
!hdfs dfs -put -f service_combined_forecast.csv /user/{username}/notebooks/adyen_downloader

In [76]:
combined_forecast_df = pd.read_csv('service_combined_forecast.csv')
print("Column names in service_combined_forecast.csv:")
print(combined_forecast_df.columns)

Column names in service_combined_forecast.csv:
Index(['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'squad', 'timestamp',
       'category'],
      dtype='object')


### Internal cases

In [77]:
def create_internal_cases_df(df):

    df['month'] = pd.to_datetime(df['month'])

    filtered_df = df[
        (df['month'].dt.date < current_month_start) &
        (df['month'].dt.date >= pd.to_datetime('2023-09-01').date()) &
        (~df['origin'].isin(['Side Conversation', 'Call'])) &
        (df['category'] == 'Internal')
    ]

    service_df = (
        filtered_df
        .groupby(['month', 'support_team'], as_index=False)
        .agg(cases=('unique_id_count', 'sum'))
        .dropna()
    )

    internal_df = service_df.rename(columns={
        'month': 'month',
        'support_team': 'squad',
        'cases': 'cases'
    })

    return internal_df

In [78]:
internal_df = create_internal_cases_df(tickets)
print(internal_df)

         month                        squad  cases
0   2023-09-01     Account Setup Operations   1915
1   2023-09-01                        China      5
2   2023-09-01                     Disputes   1171
3   2023-09-01                     Ecom APM    106
4   2023-09-01                   Ecom Cards     67
..         ...                          ...    ...
500 2026-02-01  LATAM - Operational Support    105
501 2026-02-01    LATAM - Technical Support     73
502 2026-02-01          Operational Support    569
503 2026-02-01              Partner Support     71
504 2026-02-01                    Platforms     45

[505 rows x 3 columns]


In [79]:
def prepare_forecast_df(internal_df):
    df = (
        internal_df
        .rename(columns={'month': 'ds', 'cases': 'y'})
        .dropna(subset=['ds', 'y'])
    )
    return df

In [80]:
test = prepare_forecast_df(internal_df)
print(test)

            ds                        squad     y
0   2023-09-01     Account Setup Operations  1915
1   2023-09-01                        China     5
2   2023-09-01                     Disputes  1171
3   2023-09-01                     Ecom APM   106
4   2023-09-01                   Ecom Cards    67
..         ...                          ...   ...
500 2026-02-01  LATAM - Operational Support   105
501 2026-02-01    LATAM - Technical Support    73
502 2026-02-01          Operational Support   569
503 2026-02-01              Partner Support    71
504 2026-02-01                    Platforms    45

[505 rows x 3 columns]


In [81]:
nested_df = {
    squad: df.drop(columns='squad')
    for squad, df in test.groupby('squad')
}
print(nested_df)

{'Account Setup Operations':             ds     y
0   2023-09-01  1915
15  2023-10-01  2447
30  2023-11-01  2294
44  2023-12-01  2385
59  2024-01-01  3484
74  2024-02-01  3468
90  2024-03-01  2070
105 2024-04-01  2928
121 2024-05-01  2539
137 2024-06-01  2635
153 2024-07-01  3164
169 2024-08-01  3406
186 2024-09-01  2850
202 2024-10-01  3525
220 2024-11-01  2652
238 2024-12-01  2559
255 2025-01-01  2426
273 2025-02-01  2434
291 2025-03-01  2431
309 2025-04-01  2776
327 2025-05-01  2359
344 2025-06-01  2536
361 2025-07-01  2459
379 2025-08-01  1497
397 2025-09-01  1491
415 2025-10-01  1625
433 2025-11-01  1335
451 2025-12-01  1200
469 2026-01-01  1127
487 2026-02-01  1285, 'China':             ds   y
1   2023-09-01   5
16  2023-10-01   1
45  2023-12-01   1
60  2024-01-01   1
75  2024-02-01   2
106 2024-04-01  43
122 2024-05-01  14
138 2024-06-01  31
154 2024-07-01  22
170 2024-08-01  31
187 2024-09-01  23
203 2024-10-01  15
221 2024-11-01  38
239 2024-12-01  12
256 2025-01-01   8
274 20

In [82]:
def run_internal_prophet(df):
    
     # Ensure required columns are present
    required_cols = {'ds', 'y'}
    if not required_cols.issubset(df.columns):
        return None
    
    # Remove NAs and check row count
    df = df.dropna(subset=['ds', 'y'])
    if len(df) < 10:
        return None  # Prophet requires enough data points
    
    # Fit the model with regressor
    m = Prophet()
    m.fit(df)
    
    # Make future df
    future = m.make_future_dataframe(periods=13, freq='M')
    future['ds'] = future['ds'].values.astype('datetime64[M]')
    
    # Forecast
    forecast = m.predict(future)
    last_train_date = df['ds'].max()
    forecast = forecast[forecast['ds'] > last_train_date]
    
    # Model evaluation on historical part
    comparison = df.merge(forecast[['ds', 'yhat']], on='ds', how='left').dropna()
    comparison['residuals'] = comparison['y'] - comparison['yhat']

    ss_res = np.sum(comparison['residuals'] ** 2)
    ss_tot = np.sum((comparison['y'] - np.mean(comparison['y'])) ** 2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else None

    mape = np.mean(np.abs((comparison['y'] - comparison['yhat']) / comparison['y'])) * 100

    return {
        'model': m,
        'forecast': forecast,
        'r_squared': r_squared,
        'mape': mape
    }


In [83]:
results = {}
for squad, df in nested_df.items():
    result = run_internal_prophet(df)
    if result is not None:
        results[squad] = result
print(results)

[I 2026-03-23 11:10:21,521.521 prophet] Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
[I 2026-03-23 11:10:21,521.521 prophet] Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
[I 2026-03-23 11:10:21,535.535 prophet] n_changepoints greater than number of observations. Using 23.
[D 2026-03-23 11:10:21,537.537 cmdstanpy] input tempfile: /tmp/tmpy_ntexaz/hyana4wt.json
[D 2026-03-23 11:10:21,539.539 cmdstanpy] input tempfile: /tmp/tmpy_ntexaz/_b22w3sv.json
[D 2026-03-23 11:10:21,540.540 cmdstanpy] idx 0
[D 2026-03-23 11:10:21,540.540 cmdstanpy] running CmdStan, num_threads: None
[D 2026-03-23 11:10:21,540.540 cmdstanpy] CmdStan args: ['/home/bigdata/mypkg/prophet/stan_model/prophet_model.bin', 'random', 'seed=75192', 'data', 'file=/tmp/tmpy_ntexaz/hyana4wt.json', 'init=/tmp/tmpy_ntexaz/_b22w3sv.json', 'output', 'file=/tmp/tmpy_ntexaz/prophet_modela2w151p0/prophet_model-20260323111021.csv', 'method=optimize', 'al

{'Account Setup Operations': {'model': <prophet.forecaster.Prophet object at 0x7efa374c9650>, 'forecast':            ds        trend   yhat_lower   yhat_upper  trend_lower  \
31 2026-03-01  1632.401527  1096.548876  2081.839588  1632.400497   
32 2026-04-01  1577.351556   536.840134  1529.111341  1577.348244   
33 2026-05-01  1524.077391   357.901963  1406.348120  1524.071020   
34 2026-06-01  1469.027420  1542.932515  2552.755804  1469.017932   
35 2026-07-01  1415.753255  2082.427540  3106.335370  1415.740015   
36 2026-08-01  1360.703284  1234.288872  2184.245617  1360.686560   
37 2026-09-01  1305.653313   103.950615  1073.827376  1305.631309   
38 2026-10-01  1252.379148   551.572455  1609.814805  1252.351823   
39 2026-11-01  1197.329177   240.253227  1248.029756  1197.296615   
40 2026-12-01  1144.055012   152.022842  1216.319769  1144.017186   
41 2027-01-01  1089.005041   733.123537  1759.536929  1088.961690   
42 2027-02-01  1033.955070   689.908254  1725.827912  1033.905213 

In [84]:
# Get current timestamp
current_timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Prepare lists to collect results for each component
forecast_list = []
r_squared_list = []
mape_list = []

# Loop through the results and gather data
for squad, result in results.items():
    # Add forecast data (ds and yhat)
    forecast_df = result['forecast'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    forecast_df[['yhat', 'yhat_lower', 'yhat_upper']] = forecast_df[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
    forecast_df['squad'] = squad  # Add the squad name for identification
    forecast_df['timestamp'] = current_timestamp  # Add timestamp
    forecast_df['category'] = 'Internal'  # Add category column
    forecast_list.append(forecast_df)
    
    # Add R-squared data
    r_squared_list.append({'squad': squad, 'r_squared': result['r_squared'], 'timestamp': current_timestamp})
    
    # Add MAPE data
    mape_list.append({'squad': squad, 'mape': result['mape'], 'timestamp': current_timestamp})

# Combine the lists into DataFrames
forecast_combined = pd.concat(forecast_list, ignore_index=True)
r_squared_combined = pd.DataFrame(r_squared_list)
mape_combined = pd.DataFrame(mape_list)

# Ensure correct column names
forecast_combined.columns = ['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'squad', 'timestamp', 'category']

# Save the combined data to CSV files
forecast_combined.to_csv('internal_combined_forecast.csv', mode='w', header=True, index=False)
r_squared_combined.to_csv('internal_combined_r_squared.csv', mode='w', header=True, index=False)
mape_combined.to_csv('internal_combined_mape.csv', mode='w', header=True, index=False)

print(f"All results have been saved to internal_combined_forecast.csv, internal_combined_r_squared.csv, and internal_combined_mape.csv with timestamp {current_timestamp}.")

All results have been saved to internal_combined_forecast.csv, internal_combined_r_squared.csv, and internal_combined_mape.csv with timestamp 2026-03-23 11:10:29.


In [85]:
internal_combined_forecast = pd.read_csv('internal_combined_forecast.csv')
print(internal_combined_forecast.columns)

Index(['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'squad', 'timestamp',
       'category'],
      dtype='object')


In [86]:
!hdfs dfs -put -f internal_combined_forecast.csv /user/{username}/notebooks/Workload\ Projections/Forecasted\ data

In [87]:
!hdfs dfs -put -f internal_combined_forecast.csv /user/{username}/notebooks/adyen_downloader

## Calls Forecasting

In [88]:
def prepare_forecast_df(calls, account_growth):
    
    calls['month'] = pd.to_datetime(calls['month'])
    account_growth['month'] = pd.to_datetime(account_growth['month'])
    
    calls = calls[
        (calls['month'].dt.date < current_month_start)
    ]
    
    df = (
        calls
        .merge(account_growth, on='month', how='left')
        .rename(columns={'month': 'ds', 'calls': 'y'})
        .dropna(subset=['ds', 'y', 'unique_accounts'])
    )
    return df

In [89]:
test = prepare_forecast_df(calls, combined_df)
print(test)

            ds         support_team     y  unique_accounts    type
0   2023-01-01                Japan   157           7531.0  Actual
1   2023-01-01                LATAM   100           7531.0  Actual
2   2023-01-01  Operational Support  1137           7531.0  Actual
3   2023-02-01                Japan   167           7570.0  Actual
4   2023-02-01                LATAM   107           7570.0  Actual
..         ...                  ...   ...              ...     ...
109 2026-01-01                LATAM    59           9431.0  Actual
110 2026-01-01  Operational Support  1628           9431.0  Actual
111 2026-02-01                Japan   162           9468.0  Actual
112 2026-02-01                LATAM    65           9468.0  Actual
113 2026-02-01  Operational Support  1799           9468.0  Actual

[114 rows x 5 columns]


In [90]:
nested_df = {
    squad: df.drop(columns='support_team')
    for squad, df in test.groupby('support_team')
}
print(nested_df)

{'Japan':             ds    y  unique_accounts    type
0   2023-01-01  157           7531.0  Actual
3   2023-02-01  167           7570.0  Actual
6   2023-03-01  249           7642.0  Actual
9   2023-04-01  175           7697.0  Actual
12  2023-05-01  130           7726.0  Actual
15  2023-06-01  153           7776.0  Actual
18  2023-07-01  136           7821.0  Actual
21  2023-08-01  136           7866.0  Actual
24  2023-09-01  162           7899.0  Actual
27  2023-10-01  178           7963.0  Actual
30  2023-11-01  172           8044.0  Actual
33  2023-12-01  191           8095.0  Actual
36  2024-01-01  271           8123.0  Actual
39  2024-02-01  257           8157.0  Actual
42  2024-03-01  264           8209.0  Actual
45  2024-04-01  248           8300.0  Actual
48  2024-05-01  216           8389.0  Actual
51  2024-06-01  202           8436.0  Actual
54  2024-07-01  266           8488.0  Actual
57  2024-08-01  231           8503.0  Actual
60  2024-09-01  169           8569.0  Actual


In [91]:
results = {}
for squad, df in nested_df.items():
    result = run_internal_prophet(df)
    if result is not None:
        results[squad] = result
print(results)

[I 2026-03-23 11:10:38,393.393 prophet] Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
[I 2026-03-23 11:10:38,394.394 prophet] Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
[D 2026-03-23 11:10:38,413.413 cmdstanpy] input tempfile: /tmp/tmpy_ntexaz/8pedyry6.json
[D 2026-03-23 11:10:38,417.417 cmdstanpy] input tempfile: /tmp/tmpy_ntexaz/a9449ef5.json
[D 2026-03-23 11:10:38,418.418 cmdstanpy] idx 0
[D 2026-03-23 11:10:38,418.418 cmdstanpy] running CmdStan, num_threads: None
[D 2026-03-23 11:10:38,418.418 cmdstanpy] CmdStan args: ['/home/bigdata/mypkg/prophet/stan_model/prophet_model.bin', 'random', 'seed=83367', 'data', 'file=/tmp/tmpy_ntexaz/8pedyry6.json', 'init=/tmp/tmpy_ntexaz/a9449ef5.json', 'output', 'file=/tmp/tmpy_ntexaz/prophet_modelpk451_mi/prophet_model-20260323111038.csv', 'method=optimize', 'algorithm=newton', 'iter=10000']
[I 2026-03-23 11:10:38,418.418 cmdstanpy] Chain [1] start processing
[I

{'Japan': {'model': <prophet.forecaster.Prophet object at 0x7efbe257a550>, 'forecast':            ds       trend  yhat_lower  yhat_upper  trend_lower  trend_upper  \
39 2026-03-01  261.933988  256.710109  331.216401   261.933596   261.934341   
40 2026-04-01  263.824040  227.769335  301.201420   263.822790   263.825169   
41 2026-05-01  265.653123  196.866067  272.143524   265.650725   265.655348   
42 2026-06-01  267.543175  219.806253  297.333889   267.539438   267.546664   
43 2026-07-01  269.372258  211.230071  287.555910   269.366936   269.377049   
44 2026-08-01  271.262310  207.001326  283.688857   271.254940   271.268594   
45 2026-09-01  273.152362  195.948065  270.328472   273.142937   273.160523   
46 2026-10-01  274.981445  218.968834  290.342218   274.969616   274.991587   
47 2026-11-01  276.871497  206.406208  282.263676   276.857477   276.883825   
48 2026-12-01  278.700580  233.532921  308.671045   278.683629   278.715193   
49 2027-01-01  280.590632  236.986291  313.7

[I 2026-03-23 11:10:39,255.255 cmdstanpy] Chain [1] done processing


In [92]:
# Get current timestamp
current_timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Prepare lists to collect results for each component
forecast_list = []
r_squared_list = []
mape_list = []

# Loop through the results and gather data
for squad, result in results.items():
    # Add forecast data (ds and yhat)
    forecast_df = result['forecast'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    forecast_df[['yhat', 'yhat_lower', 'yhat_upper']] = forecast_df[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
    forecast_df['squad'] = squad  # Add the squad name for identification
    forecast_df['timestamp'] = current_timestamp  # Add timestamp
    forecast_df['category'] = 'Calls'  # Add category column
    forecast_list.append(forecast_df)
    
    # Add R-squared data
    r_squared_list.append({'squad': squad, 'r_squared': result['r_squared'], 'timestamp': current_timestamp})
    
    # Add MAPE data
    mape_list.append({'squad': squad, 'mape': result['mape'], 'timestamp': current_timestamp})

# Combine the lists into DataFrames
forecast_combined = pd.concat(forecast_list, ignore_index=True)
r_squared_combined = pd.DataFrame(r_squared_list)
mape_combined = pd.DataFrame(mape_list)

# Ensure correct column names
forecast_combined.columns = ['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'squad', 'timestamp', 'category']

# Save the combined data to CSV files
forecast_combined.to_csv('calls_forecast.csv', mode='w', header=True, index=False)
r_squared_combined.to_csv('calls_r_squared.csv', mode='w', header=True, index=False)
mape_combined.to_csv('calls_mape.csv', mode='w', header=True, index=False)

print(f"All results have been saved to calls_forecast.csv, calls_r_squared.csv, and calls_mape.csv with timestamp {current_timestamp}.")

All results have been saved to calls_forecast.csv, calls_r_squared.csv, and calls_mape.csv with timestamp 2026-03-23 11:10:39.


In [93]:
!hdfs dfs -put -f calls_forecast.csv /user/{username}/notebooks/Workload\ Projections/Forecasted\ data

In [94]:
!hdfs dfs -put -f calls_forecast.csv /user/{username}/notebooks/adyen_downloader

## Copy to streams

In [5]:
!hdfs dfs -ls /streams/technical_excellence/support

NOTE: Picked up JDK_JAVA_OPTIONS:     --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED    --add-opens=java.base/java.io=ALL-UNNAMED     --add-opens=java.base/java.lang=ALL-UNNAMED     --add-opens=java.base/java.lang.reflect=ALL-UNNAMED     --add-opens=java.base/java.math=ALL-UNNAMED     --add-opens=java.base/java.net=ALL-UNNAMED     --add-opens=java.base/java.text=ALL-UNNAMED     --add-opens=java.base/java.util=ALL-UNNAMED     --add-opens=java.base/java.util.concurrent=ALL-UNNAMED     --add-opens=java.base/java.util.zip=ALL-UNNAMED     --add-opens=java.base/sun.security.util=ALL-UNNAMED     --add-opens=java.base/sun.security.x509=ALL-UNNAMED
Found 1 items
drwxr-xr-x   - catarinap hdfs          0 2026-05-07 10:59 /streams/technical_excellence/support/Workload Projections


In [96]:
# !hdfs dfs -rm -r /streams/technical_excellence/support/Workload\ Projections

In [97]:
# !hdfs dfs -cp /user/{username}/notebooks/Workload\ Projections /streams/technical_excellence/support

In [4]:
!hdfs dfs -cp -f /user/{username}/notebooks/Workload\ Projections /streams/technical_excellence/support

NOTE: Picked up JDK_JAVA_OPTIONS:     --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED    --add-opens=java.base/java.io=ALL-UNNAMED     --add-opens=java.base/java.lang=ALL-UNNAMED     --add-opens=java.base/java.lang.reflect=ALL-UNNAMED     --add-opens=java.base/java.math=ALL-UNNAMED     --add-opens=java.base/java.net=ALL-UNNAMED     --add-opens=java.base/java.text=ALL-UNNAMED     --add-opens=java.base/java.util=ALL-UNNAMED     --add-opens=java.base/java.util.concurrent=ALL-UNNAMED     --add-opens=java.base/java.util.zip=ALL-UNNAMED     --add-opens=java.base/sun.security.util=ALL-UNNAMED     --add-opens=java.base/sun.security.x509=ALL-UNNAMED
